# Lab 10 — 수렴 속도 비교하기

**확률통계 · Topic 10 · 부산대학교 정보컴퓨터공학부**

---

### 오늘의 목표

1. 표준오차가 정말 $\sigma/\sqrt{n}$ 인지 확인한다.
2. **분포에 따라 CLT 수렴 속도가 다르다**는 것을 직접 본다.
3. **Chebyshev bound**가 얼마나 느슨한지 확인하고, **Cauchy 반례**를 재현한다.

⏱ **예상 소요 시간: 35분**

> 오늘 랩은 이 과목의 중간 결산이다. Topic 1부터 봐 온 현상들이 전부 여기서 만난다.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

rng = np.random.default_rng(20260302)
print("준비 완료")

## Part 1. 표준오차는 정말 sigma/sqrt(n) 인가

**표본평균의 표준편차**를 직접 재보자. 이론값은 $\sigma/\sqrt{n}$ 이다.

### 실습 1

In [ ]:
sigma = 1.0
ns = [2, 5, 10, 20, 50, 100, 200, 500, 1000]
REPEATS = 20000

measured = []
for n in ns:
    means = rng.normal(0, sigma, size=(REPEATS, n)).mean(axis=1)
    measured.append(means.std())

# TODO 1: 이론값 sigma / sqrt(n) 을 계산하세요
theory = [0.0 for n in ns]

print(f"{'n':>6}{'측정값':>12}{'이론값':>12}")
for n, m, t in zip(ns, measured, theory):
    print(f"{n:>6}{m:12.4f}{t:12.4f}")

### 실습 2 — 로그-로그 그래프에서 기울기 확인

$\mathrm{SE} = \sigma n^{-1/2}$ 이므로 양변에 로그를 씌우면

$$\log \mathrm{SE} = \log\sigma - \tfrac{1}{2}\log n$$

**로그-로그 그래프에서 기울기가 −0.5인 직선**이 나와야 한다.

In [ ]:
plt.figure(figsize=(6.5, 4.2))
plt.loglog(ns, measured, "o", ms=8, label="measured")
plt.loglog(ns, theory, "-", lw=2, label="theory sigma/sqrt(n)")
plt.xlabel("n")
plt.ylabel("std of sample mean")
plt.title("Standard error")
plt.legend()
plt.grid(alpha=0.3, which="both")
plt.show()

# TODO 2: 로그-로그 기울기를 구하세요
#         힌트: np.polyfit(np.log(ns), np.log(measured), 1)[0]
slope = 0.0
print(f"로그-로그 기울기: {slope:.4f}   (이론값 -0.5)")

## Part 2. 분포에 따라 수렴 속도가 다르다

"$n \ge 30$ 이면 CLT를 쓸 수 있다"는 경험칙을 검증해보자.

세 분포를 비교한다.
- **Uniform** — 대칭. 빨리 수렴할 것 같다
- **Exponential** — 오른쪽으로 치우침
- **Bernoulli(p=0.02)** — 극단적으로 치우침

### 실습 3 — 표준화한 표본평균을 N(0,1)과 비교

In [ ]:
def standardized_means(sampler, mu, sigma, n, repeats=40000):
    m = sampler((repeats, n)).mean(axis=1)
    # TODO 3: 표본평균을 표준화하세요.  힌트: (m - mu) / (sigma / np.sqrt(n))
    return m


setups = [
    ("Uniform(0,1)", lambda s: rng.random(s), 0.5, np.sqrt(1 / 12)),
    ("Exponential(1)", lambda s: rng.exponential(1, s), 1.0, 1.0),
    ("Bernoulli(0.02)", lambda s: (rng.random(s) < 0.02).astype(float),
     0.02, np.sqrt(0.02 * 0.98)),
]

fig, axes = plt.subplots(3, 3, figsize=(11, 8))
xs = np.linspace(-4, 4, 200)

for i, (name, sampler, mu, sd) in enumerate(setups):
    for j, n in enumerate([5, 30, 200]):
        z = standardized_means(sampler, mu, sd, n)
        ax = axes[i, j]
        ax.hist(z, bins=60, range=(-4, 4), density=True, alpha=0.8)
        ax.plot(xs, stats.norm.pdf(xs), color="red", lw=2)
        ax.set_title(f"{name}, n={n}", fontsize=11)
        ax.set_yticks([])
plt.tight_layout()
plt.show()

🤔 **Uniform은 $n=5$ 에서도 이미 종 모양**인데,
**Bernoulli(0.02)는 $n=30$ 에서도 전혀 정규분포가 아니다.**

30개를 뽑아도 대부분 0이 나오기 때문이다 (평균 0.6개만 1).

> **"$n \ge 30$" 은 경험칙일 뿐이다.** 치우친 분포에서는 훨씬 큰 $n$ 이 필요하다.

## Part 3. Chebyshev bound는 얼마나 느슨한가

$$\mathbb{P}[|X - \mu| \ge k\sigma] \le \frac{1}{k^2}$$

### 실습 4

In [ ]:
data = {
    "Normal": rng.normal(0, 1, 300000),
    "Uniform": rng.random(300000),
    "Exponential": rng.exponential(1, 300000),
}

ks = np.arange(1, 5.01, 0.25)

plt.figure(figsize=(7.5, 4.2))
# TODO 4: Chebyshev 한계 1/k^2 를 그리세요
#         힌트: plt.plot(ks, 1 / ks ** 2, lw=3, color="red", label="Chebyshev 1/k^2")

for name, s in data.items():
    mu, sd = s.mean(), s.std()
    actual = [(np.abs(s - mu) >= k * sd).mean() for k in ks]
    plt.plot(ks, actual, "o-", ms=4, label=name)

plt.yscale("log")
plt.xlabel("k")
plt.ylabel("P[|X-mu| >= k*sigma]")
plt.title("Chebyshev bound vs actual")
plt.legend()
plt.show()

Chebyshev는 **절대 틀리지 않지만** 실제보다 훨씬 큰 값을 준다.
분포를 모를 때만 쓰는 **최후의 보험**이라고 생각하면 된다.

## Part 4. Cauchy — 조건이 깨지면

Topic 6 랩에서 본 그 분포다. 이번에는 **CLT 관점에서** 다시 본다.

### 실습 5

In [ ]:
def cauchy(size):
    return np.tan(np.pi * (rng.random(size) - 0.5))


fig, axes = plt.subplots(1, 3, figsize=(12, 3.4))
for ax, n in zip(axes, [1, 30, 1000]):
    # TODO 5: n개씩 뽑아 평균을 구하세요.  힌트: cauchy((30000, n)).mean(axis=1)
    m = cauchy(30000)
    ax.hist(m, bins=200, range=(-10, 10), density=True, color="darkorange")
    ax.set_title(f"n = {n}   (std = {m.std():.1f})")
    ax.set_xlim(-10, 10)
    ax.set_yticks([])
plt.tight_layout()
plt.show()

😲 **$n$ 을 1에서 1000으로 늘려도 히스토그램이 전혀 좁아지지 않는다.**

Uniform이나 Exponential은 $n=30$ 이면 이미 좁고 매끄러워졌는데,
Cauchy는 **1000개를 평균 내도 원래 분포와 똑같이 생겼다.**

출력된 **표준편차 값 자체도 눈여겨보자.** 418 → 44 → 418 처럼 제멋대로 튄다.
$n$ 이 커지면 줄어들어야 하는데 그렇지 않고, 실행할 때마다 값이 크게 달라진다.
**모집단의 분산이 무한대라서 표본 표준편차라는 숫자가 의미를 갖지 못하는 것**이다.
(반면 **중앙값은 셋 다 0 근처로 안정적**이다 — 중앙값은 꼬리에 휘둘리지 않는다)

> **분산이 무한대이기 때문이다.** CLT의 조건 ③이 깨졌다.
> 표준오차 $\sigma/\sqrt{n}$ 에서 $\sigma$ 자체가 존재하지 않는다.
>
> 정리에 붙은 조건은 장식이 아니다. **조건을 확인하지 않고 쓰면 이런 일이 벌어진다.**

---

## 마무리 — 자가 점검

- [ ] 표준오차가 $\sigma/\sqrt{n}$ 임을 로그-로그 기울기 −0.5로 확인했다
- [ ] 분포에 따라 CLT 수렴 속도가 다르다는 것을 보았다
- [ ] Chebyshev bound가 느슨하지만 항상 옳다는 것을 확인했다
- [ ] Cauchy에서 CLT가 실패하는 이유를 설명할 수 있다

**Topic 1 질문 — "몇 번 던져야 하나" — 에 이제 답할 수 있는가?**

> (여기에 자기 말로 써보자)

### 📌 다음 주 미니 프로젝트 2 안내가 배포됩니다